# ETL — lecturas_sensores — Mantenimiento preventivo de bomba centrífuga

**Fuente:** snapshot `data/raw/lecturas_sensores_raw.csv` (extraído de PostgreSQL).

Este notebook procesa únicamente la tabla `lecturas_sensores`: carga el CSV crudo, aplica la transformación y limpieza y exporta el dataframe limpio a `data/processed/`.

## 1. Configuración

Imports, rutas del proyecto y carga de variables de entorno.

In [17]:
from pathlib import Path
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# --- Rutas del proyecto ---
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_CATALOGO_FALLAS = PROJECT_ROOT / "data" / "raw" / "catalogo_fallas_raw.csv"
RAW_DATA_EQUIPOS = PROJECT_ROOT / "data" / "raw" / "equipos_raw.csv"
RAW_DATA_LECTURAS_SENSORES = PROJECT_ROOT / "data" / "raw" / "lecturas_sensores_raw.csv"
RAW_DATA_ORDENES_MANTENIMIENTO = PROJECT_ROOT / "data" / "raw" / "ordenes_mantenimiento_raw.csv"

PROCESSED_FILE = PROCESSED_DIR / "lecturas_sensores_clean.parquet"



## 2. Análisis, transformación y limpieza

### Tabla lecturas_sensores_raw

In [18]:
df_lecturas_sensores = pd.read_csv(RAW_DATA_LECTURAS_SENSORES)
# df_a (data frame analisis - data set de analisis) copia del dataset para mantener una copia original intacta (df)
df_lecturas_sensores_w = df_lecturas_sensores.copy()

df_lecturas_sensores_w.head()

,lectura_id,eq_id,Fecha_Hora_Reg,temp_carc,vibración_mms,pres_desc,t_amb,col_aux_sistema_v1
0,1,1,2026-08-01 00:00:00,64.08,2.39,9.90,20.25,NaN
1,2,1,2026-08-01 00:01:00,64.93,2.73,9.90,19.95,v1.0.2
2,3,1,2026-08-01 00:02:00,65.54,2.70,10.05,20.37,OK_SYS
3,4,1,2026-08-01 00:03:00,65.69,3.09,9.84,20.83,NaN
4,5,1,2026-08-01 00:04:00,65.29,2.41,9.96,19.97,NaN


In [19]:
df_lecturas_sensores_w.shape

(135130, 8)

In [20]:
df_lecturas_sensores_w.info()

<class 'pandas.DataFrame'>
RangeIndex: 135130 entries, 0 to 135129
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   lectura_id          135130 non-null  int64  
 1   eq_id               135130 non-null  int64  
 2   Fecha_Hora_Reg      135130 non-null  str    
 3   temp_carc           131146 non-null  str    
 4   vibración_mms       131142 non-null  float64
 5   pres_desc           131195 non-null  float64
 6   t_amb               131003 non-null  float64
 7   col_aux_sistema_v1  80901 non-null   str    
dtypes: float64(3), int64(2), str(3)
memory usage: 11.8 MB


In [21]:
print('nulos: ', df_lecturas_sensores_w.isna().sum().sum())
print('duplicados: ', df_lecturas_sensores_w.duplicated().sum())

nulos:  70263
duplicados:  0


In [22]:
df_lecturas_sensores_w.columns

Index(['lectura_id', 'eq_id', 'Fecha_Hora_Reg', 'temp_carc', 'vibración_mms',
       'pres_desc', 't_amb', 'col_aux_sistema_v1'],
      dtype='str')

In [23]:
df_lecturas_sensores_w = df_lecturas_sensores_w.rename(columns={
    'Fecha_Hora_Reg' : 'fecha_hora_reg',
    'vibración_mms' : 'vibracion_mms'
})

df_lecturas_sensores_w.columns

Index(['lectura_id', 'eq_id', 'fecha_hora_reg', 'temp_carc', 'vibracion_mms',
       'pres_desc', 't_amb', 'col_aux_sistema_v1'],
      dtype='str')

In [24]:
nulos = df_lecturas_sensores_w.isna().sum()
porcentaje = (df_lecturas_sensores_w.isna().mean() * 100).round(2)

pd.DataFrame({'nulos': nulos, '%': porcentaje})

,nulos,%
lectura_id,0,0.00
eq_id,0,0.00
fecha_hora_reg,0,0.00
temp_carc,3984,2.95
vibracion_mms,3988,2.95
pres_desc,3935,2.91
t_amb,4127,3.05
col_aux_sistema_v1,54229,40.13


In [25]:
df_lecturas_sensores_w = df_lecturas_sensores_w.drop(columns=['col_aux_sistema_v1'])

In [26]:
nulos = df_lecturas_sensores_w.isna().sum()
porcentaje = (df_lecturas_sensores_w.isna().mean() * 100).round(2)

pd.DataFrame({'nulos': nulos, '%': porcentaje})

,nulos,%
lectura_id,0,0.00
eq_id,0,0.00
fecha_hora_reg,0,0.00
temp_carc,3984,2.95
vibracion_mms,3988,2.95
pres_desc,3935,2.91
t_amb,4127,3.05


In [27]:
print('nulos: ', df_lecturas_sensores_w.isna().sum().sum())
print('duplicados: ', df_lecturas_sensores_w.duplicated().sum())

nulos:  16034
duplicados:  0


In [28]:
df_lecturas_sensores_w.dtypes

lectura_id          int64
eq_id               int64
fecha_hora_reg        str
temp_carc             str
vibracion_mms     float64
pres_desc         float64
t_amb             float64
dtype: object

In [29]:
# La columna mezcla dos formatos: ISO y DD/MM/YYYY HH:MM (con espacios)
fechas_iso = pd.to_datetime(
    df_lecturas_sensores_w['fecha_hora_reg'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)
fechas_local = pd.to_datetime(
    df_lecturas_sensores_w['fecha_hora_reg'].str.strip(),
    format='%d/%m/%Y %H:%M',
    errors='coerce'
)
df_lecturas_sensores_w['fecha_hora_reg'] = fechas_iso.fillna(fechas_local)

# La columna mezcla números y valores con coma decimal y símbolo °C
df_lecturas_sensores_w['temp_carc'] = (
    df_lecturas_sensores_w['temp_carc']
    .str.replace('°C', '', regex=False)
    .str.strip()
    .str.replace(',', '.', regex=False)
)
df_lecturas_sensores_w['temp_carc'] = pd.to_numeric(
    df_lecturas_sensores_w['temp_carc'],
    errors='coerce'
)


In [30]:
df_lecturas_sensores_w.dtypes

lectura_id                 int64
eq_id                      int64
fecha_hora_reg    datetime64[us]
temp_carc                float64
vibracion_mms            float64
pres_desc                float64
t_amb                    float64
dtype: object

In [31]:
print('Resumen del Dataset Work luego de transformaciones y limpieza')
print('=' * 61)
print('DF_Analysis: ', df_lecturas_sensores.shape, "-> DF_Work: ", df_lecturas_sensores_w.shape)
print('=' * 61)
print('Columnas antes: ', list(df_lecturas_sensores.columns))
print('Columnas después:', list(df_lecturas_sensores_w.columns))
print('=' * 61)
print('DF_Work tiene: ', df_lecturas_sensores_w.isna().sum().sum(), ' Valores Nulos')
print('DF_Work tiene: ', df_lecturas_sensores_w.duplicated().sum(), ' Valores Duplicados')
print('=' * 61)
print('Tipos de datos finales:')
print(df_lecturas_sensores_w.dtypes.to_string())
print('=' * 61)

Resumen del Dataset Work luego de transformaciones y limpieza
DF_Analysis:  (135130, 8) -> DF_Work:  (135130, 7)
Columnas antes:  ['lectura_id', 'eq_id', 'Fecha_Hora_Reg', 'temp_carc', 'vibración_mms', 'pres_desc', 't_amb', 'col_aux_sistema_v1']
Columnas después: ['lectura_id', 'eq_id', 'fecha_hora_reg', 'temp_carc', 'vibracion_mms', 'pres_desc', 't_amb']
DF_Work tiene:  16034  Valores Nulos
DF_Work tiene:  0  Valores Duplicados
Tipos de datos finales:
lectura_id                 int64
eq_id                      int64
fecha_hora_reg    datetime64[us]
temp_carc                float64
vibracion_mms            float64
pres_desc                float64
t_amb                    float64


## 3. Exportación a Parquet

Se guarda el dataframe limpio `df_lecturas_sensores_w` en `data/processed/lecturas_sensores_clean.parquet`.

In [32]:
# Exportación del dataframe limpio a Parquet
archivo_temporal = PROCESSED_FILE.with_suffix(".parquet.tmp")

df_lecturas_sensores_w.to_parquet(archivo_temporal, index=False)
archivo_temporal.replace(PROCESSED_FILE)

print(f"Filas: {len(df_lecturas_sensores_w):,} | Columnas: {df_lecturas_sensores_w.shape[1]}")

Filas: 135,130 | Columnas: 7
